In [ ]:
# Core libraries
import os
import sys
import time
import random
import pickle
import itertools
from collections import defaultdict

# Data manipulation and analysis
import numpy as np
import pandas as pd

# Machine learning and deep learning
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

# Chemistry libraries
from rdkit import Chem

# Custom modules
from GCN import get_smiles_dicts, get_smiles_array

# Configuration
sys.setrecursionlimit(50000)
torch.backends.cudnn.benchmark = True
torch.set_default_tensor_type('torch.cuda.FloatTensor')
torch.nn.Module.dump_patches = True

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

/usr/local/lib/python3.11/dist-packages/torch/__init__.py:614: UserWarning: torch.set_default_tensor_type() is deprecated as of PyTorch 2.1, please use torch.set_default_dtype() and torch.set_default_device() as alternatives. (Triggered internally at ../torch/csrc/tensor/python_tensor.cpp:451.)
  _C._set_default_tensor_type(t)


cuda


In [ ]:

class Fingerprint(nn.Module):
    def __init__(self, radius, T, input_feature_dim, input_bond_dim,
                 fingerprint_dim, output_units_num, p_dropout):
        super(Fingerprint, self).__init__()
        
        self.radius = radius
        self.T = T
        
        # Atom embedding
        self.atom_fc = nn.Linear(input_feature_dim, fingerprint_dim)
        self.neighbor_fc = nn.Linear(input_feature_dim + input_bond_dim, fingerprint_dim)
        self.gru_cells = nn.ModuleList([nn.GRUCell(fingerprint_dim, fingerprint_dim) for _ in range(radius)])
        self.align_layers = nn.ModuleList([nn.Linear(2 * fingerprint_dim, 1) for _ in range(radius)])
        self.attend_layers = nn.ModuleList([nn.Linear(fingerprint_dim, fingerprint_dim) for _ in range(radius)])
        
        # Molecule embedding
        self.mol_gru_cell = nn.GRUCell(fingerprint_dim, fingerprint_dim)
        self.mol_align = nn.Linear(2 * fingerprint_dim, 1)
        self.mol_attend = nn.Linear(fingerprint_dim, fingerprint_dim)
        
        self.dropout = nn.Dropout(p=p_dropout)
        self.output = nn.Linear(fingerprint_dim, output_units_num)
        self.pairwise_output = nn.Linear(fingerprint_dim * 2, output_units_num)
        
    def forward(self, atom_list1, bond_list1, atom_degree_list1, bond_degree_list1, atom_mask1,
                atom_list2, bond_list2, atom_degree_list2, bond_degree_list2, atom_mask2):
        # Process first molecule
        atom_feature1, mol_feature1 = self.process_single_molecule(
            atom_list1, bond_list1, atom_degree_list1, bond_degree_list1, atom_mask1
        )

        # Process second molecule
        atom_feature2, mol_feature2 = self.process_single_molecule(
            atom_list2, bond_list2, atom_degree_list2, bond_degree_list2, atom_mask2
        )

        # Concatenate molecular features
        combined_feature = torch.cat([mol_feature1, mol_feature2], dim=1)

        # Final pairwise prediction
        pairwise_prediction = self.pairwise_output(self.dropout(combined_feature))

        return atom_feature1, atom_feature2, pairwise_prediction

    def process_single_molecule(self, atom_list, bond_list, atom_degree_list, bond_degree_list, atom_mask):
        batch_size, mol_length, num_atom_feat = atom_list.size()
        atom_mask = atom_mask.unsqueeze(2)

        atom_feature = F.leaky_relu(self.atom_fc(atom_list))
        neighbor_feature = self.get_neighbor_feature(atom_list, bond_list, atom_degree_list, bond_degree_list, batch_size)

        attend_mask, softmax_mask = self.create_masks(atom_degree_list, mol_length)

        atom_feature = self.apply_atom_attention(atom_feature, neighbor_feature, attend_mask, softmax_mask)
        mol_feature = torch.sum(F.relu(atom_feature) * atom_mask, dim=-2)

        mol_feature = self.apply_molecule_attention(atom_feature, mol_feature, atom_mask)

        return atom_feature, mol_feature


    def get_neighbor_feature(self, atom_list, bond_list, atom_degree_list, bond_degree_list, batch_size):
        bond_neighbor = torch.stack([bond_list[i][bond_degree_list[i]] for i in range(batch_size)], dim=0)
        atom_neighbor = torch.stack([atom_list[i][atom_degree_list[i]] for i in range(batch_size)], dim=0)
        neighbor_feature = torch.cat([atom_neighbor, bond_neighbor], dim=-1)
        return F.leaky_relu(self.neighbor_fc(neighbor_feature))

    def create_masks(self, atom_degree_list, mol_length):
        attend_mask = (atom_degree_list != mol_length - 1).float().unsqueeze(-1)
        softmax_mask = torch.where(atom_degree_list == mol_length - 1, torch.tensor(-9e8).cuda(), torch.tensor(0.).cuda()).unsqueeze(-1)
        return attend_mask, softmax_mask

    def apply_atom_attention(self, atom_feature, neighbor_feature, attend_mask, softmax_mask):
        """Apply atom-level attention mechanism."""
        batch_size, mol_length = atom_feature.shape[:2]
        
        for d in range(self.radius):
            atom_feature_expand = atom_feature.unsqueeze(-2).expand(-1, -1, neighbor_feature.size(2), -1)
            feature_align = torch.cat([atom_feature_expand, neighbor_feature], dim=-1)
            
            align_score = F.leaky_relu(self.align_layers[d](self.dropout(feature_align))) + softmax_mask
            attention_weight = F.softmax(align_score, -2) * attend_mask
            
            context = torch.sum(attention_weight * self.attend_layers[d](self.dropout(neighbor_feature)), -2)
            context = F.elu(context)
            
            atom_feature = self.gru_cells[d](
                context.view(batch_size * mol_length, -1),
                atom_feature.view(batch_size * mol_length, -1)
            ).view(batch_size, mol_length, -1)
            
            neighbor_feature = F.relu(atom_feature).unsqueeze(-2).expand(-1, -1, neighbor_feature.size(2), -1)
        
        return atom_feature

    def apply_molecule_attention(self, atom_feature, mol_feature, atom_mask):
        """Apply molecule-level attention mechanism."""
        batch_size, mol_length = atom_feature.shape[:2]
        mol_softmax_mask = torch.where(atom_mask.squeeze() == 0, torch.tensor(-9e8).cuda(), torch.tensor(0.).cuda())
        
        for _ in range(self.T):
            mol_prediction_expand = mol_feature.unsqueeze(-2).expand(-1, mol_length, -1)
            mol_align = torch.cat([mol_prediction_expand, atom_feature], dim=-1)
            mol_align_score = F.leaky_relu(self.mol_align(mol_align)) + mol_softmax_mask.unsqueeze(-1)
            mol_attention_weight = F.softmax(mol_align_score, -2) * atom_mask
            
            mol_context = torch.sum(mol_attention_weight * self.mol_attend(self.dropout(atom_feature)), -2)
            mol_context = F.elu(mol_context)
            mol_feature = self.mol_gru_cell(mol_context, mol_feature)
        
        return mol_feature


class RelativeGCNModel(nn.Module):
    def __init__(self, radius, T, input_feature_dim, input_bond_dim,
                 fingerprint_dim, output_units_num, p_dropout):
        super(RelativeGCNModel, self).__init__()
        
        self.radius = radius
        self.T = T
        
        hidden_dim = fingerprint_dim * 4
        self.atom_fc = nn.Sequential(
            nn.Linear(input_feature_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, fingerprint_dim)
        )
        
        self.neighbor_fc = nn.Sequential(
            nn.Linear(input_feature_dim + input_bond_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, fingerprint_dim)
        )
        
        self.gcn_layers = nn.ModuleList([
            nn.Sequential(
                nn.Linear(fingerprint_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, fingerprint_dim)
            ) for _ in range(radius * 2)
        ])
        
        self.dropout = nn.Dropout(p=p_dropout)
        
        self.output = nn.Sequential(
            nn.Linear(fingerprint_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_units_num)
        )
    def forward(self, atom_list1, bond_list1, atom_degree_list1, bond_degree_list1, atom_mask1,
                atom_list2, bond_list2, atom_degree_list2, bond_degree_list2, atom_mask2):
        
        batch_size, mol_length, num_atom_feat = atom_list1.size()

        atom_feature1, mol_feature1 = self.process_molecule(atom_list1, bond_list1, atom_degree_list1, bond_degree_list1, atom_mask1)
        atom_feature2, mol_feature2 = self.process_molecule(atom_list2, bond_list2, atom_degree_list2, bond_degree_list2, atom_mask2)
        
        pairwise_feature = torch.cat([mol_feature1, mol_feature2], dim=1)
        pairwise_prediction = self.output(self.dropout(pairwise_feature))
        
        return atom_feature1, atom_feature2, pairwise_prediction
    def process_molecule(self, atom_list, bond_list, atom_degree_list, bond_degree_list, atom_mask):
        batch_size, mol_length, num_atom_feat = atom_list.size()
        atom_mask = atom_mask.unsqueeze(2)
        
        atom_feature = self.atom_fc(atom_list)
        
        neighbor_feature = self.get_neighbor_feature(atom_list, bond_list, atom_degree_list, bond_degree_list, batch_size, mol_length)
        
        for i, layer in enumerate(self.gcn_layers):
            neighbor_feature = layer(neighbor_feature)
            atom_feature = atom_feature + neighbor_feature
            atom_feature = self.dropout(atom_feature)
        
        mol_feature = torch.sum(F.relu(atom_feature) * atom_mask, dim=1)
        return atom_feature, mol_feature

    def get_neighbor_feature(self, atom_list, bond_list, atom_degree_list, bond_degree_list, batch_size, mol_length):
        neighbor_feature = []
        for i in range(batch_size):
            atom_degrees = atom_degree_list[i]
            bond_degrees = bond_degree_list[i]
            
            atom_degrees = torch.clamp(atom_degrees, 0, mol_length - 1)
            bond_degrees = torch.clamp(bond_degrees, 0, bond_list.size(1) - 1)
            
            atom_features = atom_list[i]
            bond_features = bond_list[i]
            
            neighbor_atoms = atom_features[atom_degrees]
            neighbor_bonds = bond_features[bond_degrees]
            
            mol_neighbor_feature = torch.cat([neighbor_atoms, neighbor_bonds], dim=-1)
            mol_neighbor_feature = self.neighbor_fc(mol_neighbor_feature)
            mol_neighbor_feature = mol_neighbor_feature.sum(dim=1)
            
            neighbor_feature.append(mol_neighbor_feature)
    
        neighbor_feature = torch.stack(neighbor_feature, dim=0)
        return F.relu(neighbor_feature)
    


In [ ]:
def create_structured_relative_ecfp_dataframes(df, holdout_size):
    """
    Create structured relative ECFP dataframes for training and testing.

    This function processes calixarene data to generate two DataFrames:
    one for training data where neither host is in the holdout set,
    and one for testing data where one host is in the holdout set for prediction.

    Parameters:
        df: Input DataFrame containing calixarene data
        holdout_size: Fraction of calixarenes to be used for testing

    Returns:
        tuple: (train_df, test_df, test_hosts_list)
    """
    split_calixarene_dict = {'predictable': ['AP1', 'AP3', 'AP4', 'AP5', 'AP6',
                                    'AP7', 'AP8', 'AP9', 'AH1', 'AH2',
                                    'AH3', 'AH4', 'AH5', 'AH6', 'AH7',
                                    'AM1', 'AM2', 'AO1', 'AO2', 'AO3',
                                    'E1', 'E3', 'E6', 'E7', 'E8', 'E11',
                                    'P-NO2', 'PSC4'],
                    'unpredictable': ['BP0', 'BP1', 'BH2', 'BM1', 'CP1',
                                      'CP2', 'DP2', 'DM1', 'DO2', 'DO3']}
    
    calixarene_df = df
    holdout_pred_amount = int(len(split_calixarene_dict['predictable']) * holdout_size)
    holdout_unpred_amount = int(len(split_calixarene_dict['unpredictable']) * holdout_size)

    holdout_calixarenes_pred = random.sample(split_calixarene_dict['predictable'], holdout_pred_amount)
    holdout_calixarenes_unpred = random.sample(split_calixarene_dict['unpredictable'], holdout_unpred_amount)
    all_holdout_calix = holdout_calixarenes_pred + holdout_calixarenes_unpred

    train_data = []
    test_data = []
    target_columns = ['H3K4', 'H3K4ac', 'H3K4me1', 'H3K4me2', 'H3K4me3', 'H3K9me3', 'H3R2me2a', 'H3R2me2s']
    test_hosts = set()

    for (idx1, row1), (idx2, row2) in itertools.permutations(calixarene_df.iterrows(), 2):
        host_pair = f"{row1['Host']},{row2['Host']}"
        smiles_pair = f"{row1['cano_smiles']},{row2['cano_smiles']}"
        
        if row1['Host'] not in all_holdout_calix and row2['Host'] not in all_holdout_calix:
            train_row = {'Host_Pair': host_pair, 'SMILES_pair': smiles_pair}
            for target in target_columns:
                train_row[target] = row1[target] - row2[target]
            train_data.append(train_row)
        else:
            if (row1['Host'] in holdout_calixarenes_pred) ^ (row2['Host'] in holdout_calixarenes_pred):
                if row1['Host'] not in holdout_calixarenes_unpred and row2['Host'] not in holdout_calixarenes_unpred:
                    test_row = {'Host_Pair': host_pair, 'SMILES_pair': smiles_pair}
                    for target in target_columns:
                        test_row[target] = row1[target] - row2[target]
                    test_data.append(test_row)
                    
                    if row1['Host'] in holdout_calixarenes_pred:
                        test_hosts.add(row1['Host'])
                    if row2['Host'] in holdout_calixarenes_pred:
                        test_hosts.add(row2['Host'])

    train_df = pd.DataFrame(train_data)
    test_df = pd.DataFrame(test_data)
    test_hosts_list = list(test_hosts)

    return train_df, test_df, test_hosts_list


def prepare_model_and_data_for_relative_learning(raw_filename, targets=None, p_dropout=0.1, fingerprint_dim=150, 
                                              weight_decay=2, learning_rate=3, radius=3, T=2):
    """
    Prepare data and model for relative learning approach with memory-efficient implementation.
    
    Parameters:
        raw_filename: Path to the CSV file containing molecular data
        targets: List of target columns to predict
        p_dropout: Dropout probability for the model
        fingerprint_dim: Dimension of molecular fingerprints
        weight_decay: Weight decay parameter for optimizer
        learning_rate: Learning rate for optimizer (as power of 10)
        radius: Radius parameter for molecular convolution
        T: Number of iterations for attention mechanism
    
    Returns:
        tuple: (model, optimizer, loss_function, feature_dicts, df, remained_df)
    """
    
    global initial_state 

    
    if targets is None:
        targets = ['H3K4', 'H3K4ac', 'H3K4me1', 'H3K4me2', 'H3K4me3', 'H3K9me3', 'H3R2me2a', 'H3R2me2s']

    feature_filename = raw_filename.replace('.csv', '.pickle')
    smiles_targets_df = pd.read_csv(raw_filename)
    smilesList = smiles_targets_df.SMILES.values
    
    remained_smiles = []
    canonical_smiles_list = []
    for smiles in smilesList:
        try:
            mol = Chem.MolFromSmiles(smiles)
            remained_smiles.append(smiles)
            canonical_smiles_list.append(Chem.MolToSmiles(Chem.MolFromSmiles(smiles), isomericSmiles=True))
        except:
            pass
    
    smiles_targets_df = smiles_targets_df[smiles_targets_df["SMILES"].isin(remained_smiles)]
    smiles_targets_df['cano_smiles'] = canonical_smiles_list

    if os.path.isfile(feature_filename):
        feature_dicts = pickle.load(open(feature_filename, "rb"))
    else:
        feature_dicts = get_smiles_dicts(smilesList)

    remained_df = smiles_targets_df[smiles_targets_df["cano_smiles"].isin(feature_dicts['smiles_to_atom_mask'].keys())]

    batch_size = 32
    n_samples = len(remained_df)
    smile_pairs = []
    relative_targets = []
    
    for i in range(0, n_samples, batch_size):
        batch_end = min(i + batch_size, n_samples)
        batch_df = remained_df.iloc[i:batch_end]
        
        for idx1, row1 in batch_df.iterrows():
            for idx2, row2 in remained_df.iterrows():
                if idx1 != idx2:
                    smile1 = row1['cano_smiles']
                    smile2 = row2['cano_smiles']
                    target1 = row1[targets].values
                    target2 = row2[targets].values
                    
                    rel_target = target1 - target2
                    smile_pairs.append(f"{smile1},{smile2}")
                    relative_targets.append(rel_target)

    df = pd.DataFrame(relative_targets, columns=targets)
    df['SMILES_pair'] = smile_pairs
    df = df[['SMILES_pair'] + targets]

    smile1, smile2 = smile_pairs[0].split(',')
    x_atom1, x_bonds1, _, _, _, _ = get_smiles_array([smile1], feature_dicts)
    num_atom_features = x_atom1.shape[-1]
    num_bond_features = x_bonds1.shape[-1]

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    loss_function = nn.MSELoss()
    model = Fingerprint(radius, T, num_atom_features, num_bond_features, fingerprint_dim, len(targets), p_dropout)
    initial_state = {k: v.clone() for k, v in model.state_dict().items()}
    model.to(device)
    optimizer = optim.Adam(model.parameters(), 10**-learning_rate, weight_decay=10**-weight_decay)


    return model, optimizer, loss_function, feature_dicts, df, remained_df


def validate(model, val_df, feature_dicts, loss_function, device, batch_size, return_predictions=False, test_smile=None):
    """
    Validate model on validation or test data.
    
    Parameters:
        model: Trained model for validation
        val_df: Validation DataFrame
        feature_dicts: Dictionary containing molecular features
        loss_function: Loss function for evaluation
        device: Computing device (CPU/GPU)
        batch_size: Size of validation batches
        return_predictions: Whether to return detailed predictions
        test_smile: Specific SMILES string to test (optional)
    
    Returns:
        tuple: Validation metrics and optionally detailed predictions
    """
    model.eval()
    total_loss = 0
    all_predictions = []
    all_true_values = []
    result_dict = {}

    with torch.no_grad():
        for i in range(0, len(val_df), batch_size):
            batch_df = val_df.iloc[i:i+batch_size]
            smile_pairs = batch_df.SMILES_pair.values
            y_val = batch_df[targets].values
            
            smiles_list1, smiles_list2 = zip(*[pair.split(',') for pair in smile_pairs])
            
            x_atom1, x_bonds1, x_atom_index1, x_bond_index1, x_mask1, _ = get_smiles_array(smiles_list1, feature_dicts)
            x_atom2, x_bonds2, x_atom_index2, x_bond_index2, x_mask2, _ = get_smiles_array(smiles_list2, feature_dicts)
            
            x_atom1, x_bonds1 = torch.Tensor(x_atom1).to(device), torch.Tensor(x_bonds1).to(device)
            x_atom_index1, x_bond_index1 = torch.LongTensor(x_atom_index1).to(device), torch.LongTensor(x_bond_index1).to(device)
            x_mask1 = torch.Tensor(x_mask1).to(device)
            
            x_atom2, x_bonds2 = torch.Tensor(x_atom2).to(device), torch.Tensor(x_bonds2).to(device)
            x_atom_index2, x_bond_index2 = torch.LongTensor(x_atom_index2).to(device), torch.LongTensor(x_bond_index2).to(device)
            x_mask2 = torch.Tensor(x_mask2).to(device)
            
            output_tuple = model(x_atom1, x_bonds1, x_atom_index1, x_bond_index1, x_mask1,
                               x_atom2, x_bonds2, x_atom_index2, x_bond_index2, x_mask2)
            predictions = output_tuple[2]
            
            loss = loss_function(predictions, torch.Tensor(y_val).to(device))
            total_loss += loss.item()
            
            all_predictions.extend(predictions.cpu().numpy())
            all_true_values.extend(y_val)
            
            if return_predictions:
                for j, smile_pair in enumerate(smile_pairs):
                    smile1, smile2 = smile_pair.split(',')
                    
                    if test_smile is None or test_smile in (smile1, smile2):
                        if smile_pair not in result_dict:
                            result_dict[smile_pair] = {}
                        
                        result_dict[smile_pair] = {
                            target: {"actual": y_val[j][k], "predicted": predictions[j][k].item()}
                            for k, target in enumerate(targets)
                        }
    
    avg_loss = total_loss / (len(val_df) // batch_size + 1)
    r2 = r2_score(all_true_values, all_predictions)
    
    if return_predictions:
        return avg_loss, r2, all_true_values, all_predictions, result_dict
    return avg_loss, r2


def train_and_evaluate(model, train_df, test_df, feature_dicts, optimizer, loss_function, num_epochs=800, batch_size=32, patience=30):
    """
    Train model using provided train and test dataframes with proper data isolation.
    
    Parameters:
        model: Neural network model to train
        train_df: Training DataFrame
        test_df: Test DataFrame
        feature_dicts: Dictionary containing molecular features
        optimizer: Optimization algorithm
        loss_function: Loss function for training
        num_epochs: Maximum number of training epochs
        batch_size: Size of training batches
        patience: Early stopping patience
    
    Returns:
        tuple: Training metrics, overall R2 score, and result dictionary
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    model.load_state_dict(initial_state) 
    optimizer = type(optimizer)(model.parameters(), **optimizer.defaults)
    
    train_subset, val_df = train_test_split(train_df, test_size=0.2)
    
    print(f"Training set: {len(train_subset)} samples")
    print(f"Validation set: {len(val_df)} samples")
    print(f"Test set: {len(test_df)} samples")
    
    best_val_loss = float('inf')
    epochs_no_improve = 0
    best_model_state = None
    
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        train_subset = train_subset.sample(frac=1).reset_index(drop=True)
        
        for i in range(0, len(train_subset), batch_size):
            batch_df = train_subset.iloc[i:i+batch_size]
            smile_pairs = batch_df.SMILES_pair.values
            y_val = batch_df[targets].values
            
            smiles_list1, smiles_list2 = zip(*[pair.split(',') for pair in smile_pairs])
            
            x_atom1, x_bonds1, x_atom_index1, x_bond_index1, x_mask1, _ = get_smiles_array(smiles_list1, feature_dicts)
            x_atom2, x_bonds2, x_atom_index2, x_bond_index2, x_mask2, _ = get_smiles_array(smiles_list2, feature_dicts)
            
            x_atom1, x_bonds1 = torch.Tensor(x_atom1).to(device), torch.Tensor(x_bonds1).to(device)
            x_atom_index1, x_bond_index1 = torch.LongTensor(x_atom_index1).to(device), torch.LongTensor(x_bond_index1).to(device)
            x_mask1 = torch.Tensor(x_mask1).to(device)
            
            x_atom2, x_bonds2 = torch.Tensor(x_atom2).to(device), torch.Tensor(x_bonds2).to(device)
            x_atom_index2, x_bond_index2 = torch.LongTensor(x_atom_index2).to(device), torch.LongTensor(x_bond_index2).to(device)
            x_mask2 = torch.Tensor(x_mask2).to(device)
            
            optimizer.zero_grad()
            output_tuple = model(x_atom1, x_bonds1, x_atom_index1, x_bond_index1, x_mask1,
                               x_atom2, x_bonds2, x_atom_index2, x_bond_index2, x_mask2)
            predictions = output_tuple[2]
            
            loss = loss_function(predictions, torch.Tensor(y_val).to(device))
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        avg_loss = total_loss / (len(train_subset) // batch_size + 1)
        
        val_loss, val_r2 = validate(model, val_df, feature_dicts, loss_function, device, batch_size)
        print(f'Epoch {epoch+1}/{num_epochs}: Train Loss={avg_loss:.4f}, Val Loss={val_loss:.4f}, Val R2={val_r2:.4f}')
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
            best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            epochs_no_improve += 1
            if epochs_no_improve == patience:
                print(f'Early stopping at epoch {epoch+1}')
                break
    
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    
    test_loss, test_r2, true_values, predictions, result_dict = validate(
        model, test_df, feature_dicts, loss_function, device, batch_size, 
        return_predictions=True
    )
    
    print(f'Final results: Val Loss={best_val_loss:.4f}, Test Loss={test_loss:.4f}, Test R2={test_r2:.4f}')
    
    overall_r2 = r2_score(true_values, predictions)
    print(f'Overall Test R2: {overall_r2:.4f}')
    
    return (best_val_loss, test_loss, test_r2), overall_r2, result_dict


def map_smiles_to_host_pairs(final_dict, remained_df):
    """
    Map SMILES pairs in the dictionary to host pair names.
    
    Parameters:
        final_dict: Dictionary with SMILES pairs as keys
        remained_df: DataFrame with 'Host' and 'cano_smiles' columns
    
    Returns:
        dict: New dictionary with host pairs as keys
    """
    smiles_to_host = dict(zip(remained_df['cano_smiles'], remained_df['Host']))
    host_pair_dict = {}
    
    for smiles_pair, values in final_dict.items():
        smiles1, smiles2 = smiles_pair.split(',')
        
        host1 = smiles_to_host.get(smiles1, "Unknown")
        host2 = smiles_to_host.get(smiles2, "Unknown")
        
        host_pair = f"{host1},{host2}"
        host_pair_dict[host_pair] = values
    
    return host_pair_dict


def process_and_format_predictions(relative_results, test_hosts, remained_df):
    """
    Process relative learning model results and format the final predictions.
    
    Args:
        relative_results: Dictionary with format {'hostA,hostB': {'target': {'actual': val, 'predicted': val}}}
        test_hosts: List of host names that are being tested
        remained_df: DataFrame with 'Host' column and individual values for each host
        
    Returns:
        Dictionary with final predictions and actual values for each test host and target
    """
    # Initialize result dictionary
    final_predictions = {host: {} for host in test_hosts}
    
    # Initialize counters to calculate averages later
    count_dict = {host: {} for host in test_hosts}
    
    # Process each host pair
    for pair, targets in relative_results.items():
        host1, host2 = pair.split(',')
        
        for target, values in targets.items():
            predicted_relative = values['predicted']
            
            # If host1 is a test host
            if host1 in test_hosts:
                # Get the value for host2 from the dataframe
                host2_value = remained_df[remained_df['Host'] == host2][target].values[0]
                
                # Calculate predicted value for host1
                host1_predicted = predicted_relative + host2_value
                
                # Add to sum for averaging later
                if target not in final_predictions[host1]:
                    final_predictions[host1][target] = host1_predicted
                    count_dict[host1][target] = 1
                else:
                    final_predictions[host1][target] += host1_predicted
                    count_dict[host1][target] += 1
            
            # If host2 is a test host
            if host2 in test_hosts:
                # Get the value for host1 from the dataframe
                host1_value = remained_df[remained_df['Host'] == host1][target].values[0]
                
                # Calculate predicted value for host2
                host2_predicted = host1_value - predicted_relative
                
                # Add to sum for averaging later
                if target not in final_predictions[host2]:
                    final_predictions[host2][target] = host2_predicted
                    count_dict[host2][target] = 1
                else:
                    final_predictions[host2][target] += host2_predicted
                    count_dict[host2][target] += 1
    
    # Calculate averages and format the output
    formatted_predictions = {}
    for host in test_hosts:
        formatted_predictions[host] = []
        for target in final_predictions[host]:
            # Calculate average prediction
            avg_prediction = final_predictions[host][target] / count_dict[host][target]
            
            # Get the actual value for the test host from the dataframe
            actual_value = remained_df[remained_df['Host'] == host][target].values[0]
            
            # Append the tuple (avg_prediction, actual_value) to the list for the host
            formatted_predictions[host].append((avg_prediction, actual_value))
    
    return formatted_predictions


def run_multiple_iterations(model, remained_df, feature_dicts, optimizer, loss_function, num_iterations=20, holdout_size=0.10, num_epochs=5, batch_size=32, patience=30):

    
    # Initialize results dictionary
    all_results = {}
    
    # Run the workflow for the specified number of iterations
    for iteration in range(num_iterations):
        print(f"Running iteration {iteration+1}/{num_iterations}")
        
        # Step 1: Create train and test dataframes
        train_df, test_df, test_hosts = create_structured_relative_ecfp_dataframes(remained_df, holdout_size)
        
        # Step 2: Train and evaluate model
        fold_results, overall_r2, final_dic = train_and_evaluate(
            model, train_df, test_df, feature_dicts, optimizer,
            loss_function, num_epochs=num_epochs, batch_size=batch_size, patience=patience
        )
        
        # Step 3: Map SMILES to host pairs
        host_pair_dict = map_smiles_to_host_pairs(final_dic, remained_df)
        
        # Step 4: Process and format predictions
        new_final = process_and_format_predictions(host_pair_dict, test_hosts, remained_df)
        
        # Store the results for this iteration
        all_results[iteration] = new_final
        
        print(f"Iteration {iteration+1} completed with overall R² = {overall_r2:.4f}")
        
    return all_results


In [8]:
targets = ['H3K4', 'H3K4ac', 'H3K4me1', 'H3K4me2', 'H3K4me3', 'H3K9me3', 'H3R2me2a', 'H3R2me2s']
holdout_size= [0.05,0.1,0.15,0.25,0.5,0.75]
model, optimizer, loss_function, feature_dicts, df , remained_df= prepare_model_and_data_for_relative_learning(
    '/notebooks/Codebase/Database/cal_abs.csv')

    
for i in holdout_size:

    all_iteration_results = run_multiple_iterations(
        model, 
        remained_df, 
        feature_dicts, 
        optimizer, 
        loss_function, 
        num_iterations=20, 
        holdout_size=i, 
        num_epochs=800, 
        batch_size=32, 
        patience=30
    )
    output = open(f'/notebooks/Codebase/Result_dict/20 split {i} AttentiveFP Relative_val.pkl', 'wb')
    pickle.dump(all_iteration_results, output)
    output.close()
 



Running iteration 1/20
Training set: 1065 samples
Validation set: 267 samples
Test set: 74 samples
Epoch 1/800: Train Loss=6.4317, Val Loss=6.4255, Val R2=0.0295
Epoch 2/800: Train Loss=5.9423, Val Loss=5.8535, Val R2=0.1205
Epoch 3/800: Train Loss=5.1001, Val Loss=5.0910, Val R2=0.2625
Epoch 4/800: Train Loss=4.3081, Val Loss=4.6556, Val R2=0.3364
Epoch 5/800: Train Loss=3.7744, Val Loss=4.1392, Val R2=0.4200
Epoch 6/800: Train Loss=3.3808, Val Loss=3.7536, Val R2=0.4658
Epoch 7/800: Train Loss=3.3764, Val Loss=3.5034, Val R2=0.5163
Epoch 8/800: Train Loss=2.9064, Val Loss=3.2031, Val R2=0.5477
Epoch 9/800: Train Loss=2.6288, Val Loss=2.6072, Val R2=0.6258
Epoch 10/800: Train Loss=2.3493, Val Loss=2.4985, Val R2=0.6379
Epoch 11/800: Train Loss=2.2285, Val Loss=2.0165, Val R2=0.7065
Epoch 12/800: Train Loss=1.7656, Val Loss=1.7794, Val R2=0.7525
Epoch 13/800: Train Loss=1.7090, Val Loss=1.5840, Val R2=0.7736
Epoch 14/800: Train Loss=1.4677, Val Loss=1.3617, Val R2=0.8106
Epoch 15/800: 